In [2]:

import sys, os
sys.path.append(os.path.abspath("..")) 
import json
import torch
from datasets import GANDataset
from models import Generator
from utils import args_gan, gen_image, plotly_generate

**Read configuration files and arguments:**

In [3]:
# Arguments
parser = args_gan()
args, unknown = parser.parse_known_args()

args.particle = "proton_contained"
args.metadata_path = "/scratch/libota/sfgd_va_nn_data/NN_Data/metadata.pkl"
args.dataset_path = "/scratch/libota/sfgd_va_nn_data/NN_Data/{}/{}/{}.npz"
args.gan_ind_path = "/scratch/libota/sfgd_va_nn_data/NN_Data/gan_ind.pkl"
args.save_dir = "/scratch2/libota/SFGD_Vertex_Activity/Results/gan/"
args.checkpoint_path = "/scratch2/libota/SFGD_Vertex_Activity/Results/gan/checkpoints"
args.checkpoint_name = "proton_contained"

args.epochs = 50
args.log_every_n_steps = 2000
args.batch_size = 3072
args.hidden = 64
args.warmup_steps = 10
args.num_workers = 64

**Load the pre-trained weights of the different generative-adversarial-network (GAN) models:**

In [3]:
print(checkpoint_p['state_dict'].keys())

NameError: name 'checkpoint_p' is not defined

In [4]:
# Dataset and generator models
test_set_p = GANDataset(args, split="test")

# Geneator and critic models
generator = Generator(input_size=args.input_size, label_size=args.label_size, noise_size=args.noise_size,
                          hidden=args.hidden, n_layers=args.layers, attn_heads=args.attn_heads, dropout=args.dropout)

checkpoint_path = "/".join((args.checkpoint_path, args.checkpoint_name, "val_loss", "epoch_9_v1.ckpt"))
# Load weights of pre-trained generator models
checkpoint_p = torch.load(checkpoint_path, map_location='cpu')

state_dict = {
    key.replace("generator.", ""): value for key, value in checkpoint_p['state_dict'].items()
}

generator.load_state_dict(state_dict, strict=False)
generator.eval();

**Run each GAN on some arbitrary input kinematics:**

In [5]:
'''
Proton GAN
'''
import numpy as np

# Set your kinematics here:
# ke = 30.3  # Initial kinetic energy
# ini_dir = [0.9999999999999999, 0.0, 0.0]  # Initial direction
# ini_pos = [-1.5, -4.2, 2.7]  # Initial 3D position (mm)

# get one event from the test set
event = test_set_p[100]
# convert torch tensor to numpy array
ke = float(event['ke'].numpy())
ini_pos = event['pos_ini'].numpy()
ini_dir = event['dir_ini'].numpy()
img = event['image'].numpy()

print(ke, ini_pos, ini_dir, img)

params = np.array([ini_pos[0], ini_pos[1], ini_pos[2], ke, ini_dir[0], ini_dir[1], ini_dir[2]])


# Run the generator!
generated_p = gen_image(generator=generator, args=args, test_set=test_set_p, 
                        ke=ke, ini_dir=ini_dir, ini_pos=ini_pos)[0]



1.215614395985719 [ 0.17960486 -0.26424485 -0.06899551] [ 0.47998244 -0.13974476  0.8660764 ] [0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.01737812 0.15850362 0.09091399
 0.         0.09889295 1.23615666 0.03324926 0.0316512  0.
 0.         0.         0.         0.         0.         0.
 0.         0.         0.         0.         0.         0.
 0.10589885 0.20312009 0.         0.         0.22331144 2.57890533
 2.40119034 0.         0.         0.63224962 0.2818126  0.118564

/tmp/ipykernel_4122811/131756782.py:14: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  ke = float(event['ke'].numpy())


**Visualise the GAN-generated images:**

In [ ]:
'''
Plot the generated images!
'''


#generated_p = generated_p.numpy()
# copy the image to a pure numpy array

print(generated_p.shape)

generated_plot = np.zeros((5, 5, 5))

for i in range(generated_plot.shape[0]):
    for j in range(generated_plot.shape[1]):
        for k in range(generated_plot.shape[2]):
            generated_plot[i, j, k] = float(generated_p[i, j, k])
print(generated_plot)
# check the type of the elements in the array
print(generated_plot.dtype)

# Max deposited energy in one voxel
max_energy = generated_plot.max()

#generated_plot[generated_plot > 150] = 0
generated_plot[generated_plot < 5] = 0
print(max_energy)




torch.Size([5, 5, 5])
[[[  3.76452136   3.49148726   4.14442205   5.12487888   7.09175587]
  [  3.32722759   3.44210267   3.51010442   6.29031134   5.67046356]
  [  3.5499835    3.27349043   5.08540869   3.60891676   3.82060003]
  [  3.09173965   2.97639203   3.35682273   3.31637836   3.9067874 ]
  [  3.30163288   3.18121243   3.47096419   3.51345372   3.37787127]]

 [[  3.26511216   3.70739818  16.81232262  17.73688698  33.16010284]
  [  3.15198421   3.20419407   8.93131351  19.67315483  38.12617111]
  [  3.29608846   3.4729526   11.70130157  11.08435631  10.90658283]
  [  2.82832074   2.81795788   3.65669513   3.74063468   4.28839302]
  [  3.16075063   3.08901381   3.45099211   3.43469143   3.89178538]]

 [[  4.24254417   3.98166871  49.9737587   68.90126801   4.44721174]
  [  3.43509626   8.33751678  80.23310089 214.42640686  12.12389565]
  [  8.27472973  11.43241596 303.05926514  55.67251205  33.60361481]
  [  3.06055021   3.55067229  11.46187592  11.76512909  13.29595566]
  [  3.3

In [14]:
generated_plot = generated_plot.astype(np.float32)

In [7]:

print("- Proton image:")
plotly_generate(generated_plot, max_energy=max_energy)


- Proton image:


In [8]:
event = test_set_p[100]
true_img = np.zeros((125,))

for i in range(true_img.shape[0]):
    true_img[i] = float(event["image"][i])

true_img = true_img.reshape(5, 5, 5)

# get back the normalization
true_img *= test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std']

print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['std'])
print(test_set_p.metadata['statistics']['per_tree']['proton_contained']['recon_charge']['mean'])

max_energy = true_img.max()

print(true_img)

print("- True image:")
plotly_generate(true_img, max_energy=max_energy)



303.3067761656923
197.1132585084357
[[[  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]]

 [[  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]
  [  0.           0.           0.           0.           0.        ]]

 [[  0.           0.           0.           0.           0.        ]
  [  0.           0.           5.27090168  48.07522202  27.5748291 ]
  [  0.          29.99490166 374.93469238  10.08472633   9.60002422]
  [  0.           0.           0.           0.           0.    

In [16]:
print(event["image"].numpy())

[ 197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  266.32901016  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  197.11325851  197.11325851  197.11325851
  197.11325851  197.11325851  261.34530654  197.11325851  197.11325851
  197.11325851  263.76361099 1348.25107589  298.85057845  197.11325851
  197.11325851  197.11325851  286.39696517  197.11325851  197.11325851
  197.